### Maximal Marginal Relevance
MMR(Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

In [3]:
# Step 1: Load and chunk the document
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.', metadata={'source': 'langchain_rag_dataset.txt'}),
 Document(page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.', metadata={'source': 'langchain_rag_dataset.txt'}),
 Document(page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.', metadata={'source': 'langchain_rag_dataset.txt'}),
 Document(page_conten

In [4]:
# Step 2: Use FAISS Vector Store with HuggingFace Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embedding_model)
vector_store

c:\RAG\env_langchain_lessthan1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\RAG\env_langchain_lessthan1\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
# Step 3: Create MMR Retrieval Chain
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000028CDC05CDD0>, search_type='mmr', search_kwargs={'k': 3})

In [6]:
# Step 4: Create a prompt template for the LLM
prompt_template = PromptTemplate.from_template("""
Answer the question based on the context provided.
                                               
Context:
{context}
                                               
Question:
{input}
""")

In [7]:
# Step 5: Initialize the LLM
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",  # correct argument in <1.0
    temperature=0.2
)

c:\RAG\env_langchain_lessthan1\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [8]:
# Step 6: RAG Pipeline
# Create a chain that combines retrieved documents and the LLM to answer questions
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

In [9]:
# Step 7: Query
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("Answer:\n", response["answer"])

Answer:
 LangChain supports agents by allowing them to interact with external APIs and databases, as well as use tools like calculators, search APIs, or custom functions based on instructions. It also supports memory by enabling models to retain previous interactions, making multi-turn conversations more coherent. Additionally, LangChain supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory.


In [10]:
# To get Relevant chunks that we got from MMR
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.', metadata={'source': 'langchain_rag_dataset.txt'}),
  Document(page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.', metadata={'source': 'langchain_rag_dataset.txt'}),
  Document(page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory.', metadata={'source': 'langchain_rag_d